<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/basic_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/vishaljoshi24/dnd-concordia/main/examples/requirements.in' 'git+https://github.com/vishaljoshi24/dnd-concordia.git#egg=gdm-concordia'
  %pip list

In [ ]:
!pip install gdm-concordia[ollama]
# !pip install gdm-concordia[mistral]

Dependencies

In [ ]:
from collections.abc import Mapping
import dataclasses
from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.components.agent import situation_representation_via_narrative
from concordia.document import interactive_document
from concordia.language_model import language_model
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

In [ ]:
API_TYPE = 'ollama'
MODEL_NAME = 'qwen3:8b'
API_KEY = '3ef69ef7982642a488d7044453dcbf39.1-F9a-ZhOw4L7zvc8QMlQqMB'
DISABLE_LANGUAGE_MODEL = False

In [ ]:
model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
    # api_key=API_KEY
)

In [ ]:
DISABLE_LANGUAGE_MODEL = False
if not DISABLE_LANGUAGE_MODEL and not API_KEY:
  raise ValueError('API_KEY is required.')
if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

In [ ]:
prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

In [ ]:
# Copyright 2023 DeepMind Technologies Limited.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Component that provides the default role playing instructions to an agent."""

from concordia.components.agent import constant

DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL = 'Role playing instructions'


class Instructions(constant.Constant):
  """A component that provides the role playing instructions for the agent."""

  def __init__(
      self,
      agent_name: str,
      pre_act_label: str = DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL,
  ):
    state = (
        # f'The instructions for how to play the role of {agent_name} are as '
        # 'follows. This is a social science experiment studying how well you '
        # f'play the role of a character named {agent_name}. The experiment '
        # 'is structured as a tabletop roleplaying game (like dungeons and '
        # 'dragons). However, in this case it is a serious social science '
        # 'experiment and simulation. The goal is to be realistic. It is '
        # f'important to play the role of a person like {agent_name} as '
        # f'accurately as possible, i.e., by responding in ways that you think '
        # f'it is likely a person like {agent_name} would respond, and taking '
        # f'into account all information about {agent_name} that you have. '
        # 'Always use third-person limited perspective.'

        # Baseline D&D Instructions

        f'The instructions for how to play the role of {agent_name} are as '
        'follows. This is a short scenario in which you '
        f'play the role of a character named {agent_name}. The experiment '
        'is structured as a tabletop roleplaying game (like dungeons and '
        'dragons). The goal is to be consistent, but creative. It is '
        f'important to play the role of a person like {agent_name} as '
        f'accurately as possible, i.e., by responding in ways that you think '
        f'it is likely a person like {agent_name} would respond, and taking '
        f'into account all information about {agent_name} that you have. '
        'It is important that you collaborate with with the other players'
        'on the task at hand and follow the Game Master instructions.'
        'Always use first-person limited perspective.'

        # # D&D Instructions with few-shot examples

        # f'The instructions for how to play the role of {agent_name} are as '
        # 'follows. This is a short scenario in which you '
        # f'play the role of a character named {agent_name}. The experiment '
        # 'is structured as a tabletop roleplaying game (like dungeons and '
        # 'dragons). The goal is to be consistent, but creative. It is '
        # f'important to play the role of a person like {agent_name} as '
        # f'accurately as possible, i.e., by responding in ways that you think '
        # f'it is likely a person like {agent_name} would respond, and taking '
        # f'into account all information about {agent_name} that you have. '
        # 'It is important that you cooperate with with the other players,'
        # 'on the task at hand and follow the Game Master instructions.'
        # 'An example of cooperative strategising is the following: "Travis: before\n'
        # 'we go in, are there any bits of whitestone on the ground, like'
        # 'stuff in chunks or small pieces or is it pretty clean-ish?"'
        # 'Another example of cooperative strategising is "Marisha: Also, doesn\'t\n'
        # 'Always use first-person limited perspective.'
    )
    super().__init__(state=state, pre_act_label=pre_act_label)

D&D Agent Prefab

In [ ]:
import dataclasses
from concordia.typing import prefab as prefab_lib
from concordia.agents import entity_agent_with_logging
from concordia.components import agent as agent_components

@dataclasses.dataclass
class DnDAgent(prefab_lib.Prefab):
    description: str = "A custom agent that acts like a D&D player."

    params: Mapping[str, str] = dataclasses.field(
      default_factory=lambda: {
          'name': '',
          'goal': '',
          'dnd_class': '',
          'dnd_race': '',
          'background': '',
          'personality': '',
          'ideals': '',
          'bonds': '',
          'flaws': '',
      }
    )

    def build(self, model, memory_bank):
        name = self.params.get("name", "Agent")
        goal = self.params.get("goal", "")
        dnd_class = self.params.get("class", "")
        dnd_race = self.params.get("race", "")
        background = self.params.get("background", "")
        personality_traits = self.params.get("personality", "")
        ideals = self.params.get("ideals", "")
        bonds = self.params.get("bonds", "")
        flaws = self.params.get("flaws", "")

        # Create components
        memory = agent_components.memory.AssociativeMemory(
            memory_bank=memory_bank)
        instructions = Instructions(
            agent_name=name)
        observation = agent_components.observation.LastNObservations(
            history_length=50)

        components = {
            agent_components.memory.DEFAULT_MEMORY_COMPONENT_KEY: memory,
            "Instructions": instructions,
            agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY: observation,
        }

        act_component = agent_components.concat_act_component.ConcatActComponent(
            model=model,
            component_order=list(components.keys()),
        )

        return entity_agent_with_logging.EntityAgentWithLogging(
            agent_name=name,
            act_component=act_component,
            context_components=components,
        )

# Register custom prefab

In [ ]:
prefabs['dndagent__Entity'] = DnDAgent()

Context, Shared Memories, Player Agents and Game Masters

In [ ]:
RAT_INFESTATION_CONTEXT = ('This D&D short scenario specifically concerns a rat infestation.',
'It is set in the craft brewery Company and is in dire need of help.',
'Alice and Bob are here to sort out a RAT INFESTATION in the brewery\'s BASEMENT.',
'For the duration of the scenario, only Alice, Bob and the brewery owner Charlie are in the brewery',
'At the beginning of this adventure Alice and Bob',
'meet in the Wizard Tower Brewing Company. These two adventurers',
'DO NOT know each other AT FIRST and need to get to know each other.',
'The Charlie hands out pints of Ale to Alice and Bob as they get to know each other.',)

RAT_INFESTATION_MEMORIES =  ('Charlie the brewery owner gives Alice and Bob a run-down of the task with some background:',
'- The business has been doing well and looks to expand its operations.',
'But first the beer cellar needed to be expanded.',
'- Workmen that he sent down into the cellar to expand it were attacked by',
'large black rats which came out of the wall they were digging out.',
'- The workmen escaped unharmed but the cellars are unusable which is bad for business.',
'- The rats may have something to do with the wizard\'s tower on the site which the brewery',
'takes its name from.',
'The brewery owner then lays out the terms of the task:',
'- The party must dispose of the rats.',
'- They must discover the origin of the rats and make sure they permanently',
'stop the infestation.',
'- They will each be rewarded 25 gold coins.',)

MISSING_CHILD_CONTEXT = (
    'This D&D short campaign specifically concerns a missing otter called Erin, who is the daughter of Diane. Diane is a magical walrus.',
    'Diane has brought a group of animals to the SPINE OF THE WORLD mountain range, hoping to protect them from being used by an evil druid to wreak havoc on ICEWIND DALE.',
    'She requests the help of Alice and Bob to stem the threats against everyone.',
    'At the beginning of this adventure Alice and Bob meet Diane'
    'at her cave in the SPINE OF THE WORLD mountain range. Alice and Bob'
    'DO NOT know each other AT FIRST and need to get to know each other.'
)

MISSING_CHILD_MEMORIES = (
    'Alice and Bob get to know each other after being rescued from an avalanche. Their rescuer is a giant muskrat in service to Diane, a magical walrus.',
    'They meet Diane in a cavern through which the REDRUN river flows back out to the open air. Diane asks Alice and Bob to save her child Erin, a magical otter that has been spying on the city of EASTHAVEN.',
    'Alice and Bob have been tasked with first finding Erin. Erin was last spotted near EASTHAVEN which Alice and Bob can get to by following the REDRUN.',
)

In [ ]:
from concordia.typing import scene as scene_lib
from concordia.typing import entity as entity_lib

querying_scene = scene_lib.SceneTypeSpec(
    name='querying',
    game_master_name='conversation rules',
    # action_spec=entity_lib.free_action_spec(
    #     call_to_action=entity_lib.DEFAULT_CALL_TO_SPEECH
    # ),
    action_spec=entity_lib.free_action_spec(
        # call_to_action='Based on the information you already have, you should question the brewery owner about the rat infestation. Do not repeat the other agent\'s questions.'
        call_to_action='Based on the information you already have, you should question Diane about Erin and her mission. Do not repeat the other agent\'s questions.'
    ),
)

planning_scene = scene_lib.SceneTypeSpec(
    name='planning',
    game_master_name='conversation rules',
    # action_spec=entity_lib.free_action_spec(
    #     call_to_action=entity_lib.DEFAULT_CALL_TO_SPEECH
    # ),
    action_spec=entity_lib.free_action_spec(
        # call_to_action='Based on the questions you\'ve asked, you should plan how to clear the rat infestation. Come up with different plans or contribute to the same plan with the other agent. Do not repeat each other.'
        call_to_action='Based on the questions you\'ve asked, you should plan how to search for Erin. Come up with different plans or contribute to the same plan with the other agent. Do not repeat each other.'
    ),
)

investigation_scene = scene_lib.SceneTypeSpec(
    name='investigating',
    game_master_name='conversation rules',
    # action_spec=entity_lib.free_action_spec(
    #     call_to_action=entity_lib.DEFAULT_CALL_TO_SPEECH
    # ),
    action_spec=entity_lib.free_action_spec(
        # call_to_action='Based on your plan, you should explore the brewery\'s basement. Act consistently according to your plan.'
        call_to_action='Based on you plan, you should search the mountain range for Erin. Act consistently according to your plan.'
    ),
)

In [ ]:
scenes = [
    scene_lib.SceneSpec(
        scene_type=querying_scene,
        participants=['Alice', 'Bob'],
        num_rounds=5,
        # premise={'Alice': ['You are questioning Charlie and getting to know Bob.'],
        #          'Bob': ['You are questioning Charlie and getting to know Alice.'],
                #  },
        premise={'Alice': ['You are questioning Diane and getting to know Bob.'],
                 'Bob': ['You are questioning Diane and getting to know Alice.'],
                 },
    ),
    scene_lib.SceneSpec(
        scene_type=planning_scene,
        participants=['Alice', 'Bob'],
        num_rounds=5,
        # premise={'Alice': ['Plan how to clear out the rats, with Bob.'],
        #          'Bob': ['Plan how to clear out the rats, with Alice.']},
        premise={'Alice': ['Plan how to search for Erin, with Bob.'],
                 'Bob': ['Plan how to search for Erin, with Alice.']},
    ),
    scene_lib.SceneSpec(
        scene_type=investigation_scene,
        participants=['Alice', 'Bob',],
        num_rounds=5,
        # premise={'Alice': ['Explore the basement of the brewery with Bob.'],
        #          'Bob': ['Explore the basement of the brewery with Alice.']},
        premise={'Alice': ['Search the mountain range for Erin with Bob.'],
                 'Bob': ['Search the mountain range for Erin with Alice.']},
    ),
]

In [ ]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Alice',
            'goal': 'To cooperate with Bob in completing the task given by the brewery owner',
            'class': 'Warlock',
            'race': 'Mountain Dwarf',
            'background': 'sage',
            'personality_traits': 'There is nothing I like more than a good mystery. I am willing to listen to every side of an argument before I make my own judgement.',
            'ideals':'No limits. Nothing should fetter the infinite possibility inherent in all existence.',
            'bonds': 'I have been searching my whole life for the answer to a certain question.',
            'flaws': 'I am easily distracted by the promise of information.'
            },
    ),
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Bob',
            'goal': 'To cooperate with Alice in completing the task given by the brewery owner',
            'class': 'Bard',
            'race': 'Human',
            'background': 'folk hero',
            'personality_traits': 'If someone is in trouble, I am always ready to lend help. I judge people by their actions, not their words.',
            'ideals': 'Respect. People deserve to be treated with dignity and respect.',
            'bonds': 'I protect those who cannot protect themselves.',
            'flaws': 'I have a weakness for the vices of the city, especially hard drink.',
            },
    ),
    prefab_lib.InstanceConfig(
        prefab='dialogic_and_dramaturgic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
            'scenes': scenes,
            'acting_order': 'game_master_choice',
            'shared_memories': MISSING_CHILD_MEMORIES,
        },
    ),
]

In [ ]:
config = prefab_lib.Config(
    default_premise=(MISSING_CHILD_CONTEXT),
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

Instantiate and Run Simulation

In [ ]:
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [ ]:
results_log = runnable_simulation.play(max_steps=15)

Logging

In [ ]:
from concordia.utils.structured_logging import AIAgentLogInterface

interface = AIAgentLogInterface(results_log)

In [ ]:
overview = interface.get_overview()

In [ ]:
print(overview)

In [ ]:
actions = interface.get_component_values()
for action in actions:
    print(f"Step {action['step']} [{action['entity_name']}]: {action['value']}")

# Filter by entity and step range
# interface.get_component_values(
#     entity_name='Alice', step_range=(1, 5),
# )

In [ ]:
import json
from concordia.utils.structured_logging import SimulationLog

# Save
with open('/simulation_log.json', 'w') as f:
    f.write(results_log.to_json())

In [ ]:
log = SimulationLog.from_json(open('/simulation_log.json').read())
interface = AIAgentLogInterface(log)